# Hunyuan3D-2.1：正面圖批次產生 3D 形狀（Kaggle TPU v5e-8）

3D 流程的**前半段**：用 Hunyuan3D-2.1（目前最強的開源形狀模型之一）從正面圖做出白模。**貼圖**交給 `hunyuan3d2_kaggle_2xt4.ipynb` 完成。

- **8 顆 TPU 同時做**：每顆晶片跑一個 worker，從同一個 `in/` 各自拿圖，不會重複。模型以 bf16 執行，網格用 `QUALITY` 指定的解析度（high：512）。
- **輸入**：跟 3D 流程相同的檔名規則（`character-xxx-front.png`、`weapon-xxx-stuff.png`…）。圖可以來自 Drive 分享連結（`IN_FOLDER_URL`）或 Kaggle Dataset。
- **輸出**：`/kaggle/working/hunyuan3d-shapes/`。`work/<名稱>/` 裡有 `front.png`（去背後的正面圖）、`shape.glb`（白模）和 `shape_import.json`；`in/` 是輸入圖的副本。另外打包成一個 zip。
- **接著貼圖**：打開 `hunyuan3d2_kaggle_2xt4.ipynb`，Add Input 加入**這份 notebook 的 Output**，再執行。它會自動匯入這裡的白模，跳過視角和形狀階段，直接貼圖、輸出 GLB。`in/` 的副本也會被讀到，所以不用再提供一次圖。
- **類別**：Hunyuan3D-2.1 只看正面圖。monster、bigobject 需要背面圖，預設不在 `CATEGORIES` 裡，留給 T4 的 Hunyuan3D-2mv 處理。

**每次開始前**（右側 Session options）：Accelerator 選 **TPU v5e-8**，開啟 **Internet**；用 Dataset 提供圖時，Add Input 加入那個 Dataset。

**背景執行**：右上 **Save Version → Save & Run All (Commit)**，關掉瀏覽器也會繼續跑。TPU 單次最多約 9 小時；到 `STOP_AFTER_HOURS` 就不再開始新的圖。

**限制**：PyTorch/XLA 在 TPU 上跑 Hunyuan3D-2.1 不是官方支援的組合，所以區塊 1 會先做 smoke test。每顆晶片的第一張圖要先編譯，會多花幾分鐘。

In [ ]:
# 1. Check TPU, RAM, disk and Internet; install PyTorch/XLA; run a TPU smoke test
import os, platform, shutil, socket, subprocess, sys, time

SESSION_START = time.time()  # Kaggle stops a TPU session about 9 hours after it starts.
print(f"Python: {platform.python_version()}")

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    res = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    if res.returncode != 0:
        print(res.stdout[-3000:], res.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return res.stdout

if not (os.environ.get("TPU_ACCELERATOR_TYPE") or os.path.exists("/dev/vfio") or any(n.startswith("accel") for n in os.listdir("/dev"))):
    raise RuntimeError("No TPU. Session options > Accelerator > TPU v5e-8.")
print(f"TPU: {os.environ.get('TPU_ACCELERATOR_TYPE', '(TPU device found)')}")

RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
# Each worker keeps about 6 GB of RAM (image encoder, grid, mesh cleanup); loading takes 20 GB, one at a time.
WORKERS = max(1, min(8, int((RAM_GB - 24) // 6)))
print(f"RAM: {RAM_GB:.0f} GB -> {WORKERS} worker(s), one per TPU chip")

candidates = ["/tmp", "/kaggle/tmp"]
for d in candidates:
    os.makedirs(d, exist_ok=True)
WORK = max(candidates, key=lambda d: shutil.disk_usage(d).free)
print(f"Scratch: {WORK} ({shutil.disk_usage(WORK).free / 1024**3:.0f} GB free)")

try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
except OSError:
    raise RuntimeError("No internet. Session options > Internet on.")

# PyTorch/XLA 2.8 is the version reported to work on Kaggle's v5e-8 (2.9 fails loading large models on v5e).
TORCH_PINS = ["torch==2.8.0", "torchvision==0.23.0", "torch_xla[tpu]==2.8.0"]
have = subprocess.run([sys.executable, "-c", "import torch, torch_xla; print(torch.__version__, torch_xla.__version__)"],
                      capture_output=True, text=True).stdout.split()
if len(have) != 2 or not all(v.startswith("2.8.0") for v in have):
    sh(f"{sys.executable} -m pip install -q " + " ".join(f"'{p}'" for p in TORCH_PINS)
       + " -f https://storage.googleapis.com/libtpu-releases/index.html")

# Only one process can hold the TPU: the smoke test runs in a child process, the notebook never touches the TPU.
SMOKE = r"""
import torch, torch_xla, torch_xla.runtime as xr
import torch_xla.distributed.xla_multiprocessing as xmp

def check(index):
    dev = torch_xla.device()
    a = torch.randn(1024, 1024, dtype=torch.bfloat16)
    b = torch.randn(1024, 1024, dtype=torch.bfloat16)
    c = (a.to(dev) @ b.to(dev)).cpu().float()
    ref = a.float() @ b.float()
    rel = ((c - ref).abs().max() / ref.abs().max()).item()
    assert rel < 0.05, rel
    print(f"TPU_OK chip={index} type={xr.device_type()} rel_err={rel:.4f}", flush=True)

if __name__ == "__main__":
    xmp.spawn(check, start_method="spawn")
"""
smoke_path = os.path.join(WORK, "tpu_smoke.py")
with open(smoke_path, "w") as f:
    f.write(SMOKE)
res = subprocess.run([sys.executable, smoke_path], text=True, capture_output=True,
                     env=dict(os.environ, PJRT_DEVICE="TPU"), timeout=15 * 60)
print(res.stdout.strip())
chips = res.stdout.count("TPU_OK")
if res.returncode != 0 or chips == 0:
    print(res.stderr[-4000:])
    raise RuntimeError("TPU smoke test failed: PyTorch/XLA cannot run on this TPU. See the error above.")
WORKERS = min(WORKERS, chips)
print(f"TPU smoke test passed on {chips} chip(s); using {WORKERS} worker(s).")

In [ ]:
# 2. Settings and input images
# A Drive folder shared as "Anyone with the link" (at most 50 files), or empty to use attached datasets only.
IN_FOLDER_URL = ""
# Hunyuan3D-2.1 builds from the front image only. monster and bigobject need their back view, so by default they
# are left to the 2x T4 notebook (Hunyuan3D-2mv). Add them here to build them from the front alone.
CATEGORIES = ["character", "armor", "appearance", "weapon", "equipment", "prop", "environment", "object"]
QUALITY = "high"  # high: octree 512 and more faces for every category. standard: the Colab settings (mostly 384).
SHAPE_STEPS = 50
SHAPE_GUIDANCE = 5.0
SHAPE_SEED = 1234
# No new image starts after this many hours, leaving time to pack results before the session ends.
STOP_AFTER_HOURS = 8
PIPELINE_REF = "claude/gallant-thompson-py38mf"
HY21_COMMIT = "82920d643c0dc2f7bfd7255f45f62d386edfe60c"  # Tencent-Hunyuan/Hunyuan3D-2.1

if "SESSION_START" not in globals():
    raise RuntimeError("Run block 1 first.")
import importlib, os, subprocess, sys

PIPELINE_DIR = os.path.join(WORK, "colab-ImageTo3D")
if not os.path.exists(PIPELINE_DIR):
    sh(f"git clone -q https://github.com/chousir/colab-ImageTo3D.git {PIPELINE_DIR}")
sh(f"git fetch -q origin {PIPELINE_REF} && git checkout -q -B run FETCH_HEAD", cwd=PIPELINE_DIR)
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)
import imageto3d.config, imageto3d.assets, imageto3d.kaggle_io
for module in (imageto3d.config, imageto3d.assets, imageto3d.kaggle_io):
    importlib.reload(module)

DRIVE_ROOT = "/kaggle/working/hunyuan3d-shapes"  # only /kaggle/working becomes the notebook's output
for name in ("in", "work"):
    os.makedirs(os.path.join(DRIVE_ROOT, name), exist_ok=True)
sources = ["/kaggle/input"]
if IN_FOLDER_URL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown==5.2.0"], check=True)
    import gdown
    gdown.download_folder(IN_FOLDER_URL, output=os.path.join(WORK, "drive_in"), quiet=True, resume=True)
    sources.insert(0, os.path.join(WORK, "drive_in"))
copied = imageto3d.kaggle_io.collect_inputs(sources, os.path.join(DRIVE_ROOT, "in"))
print(f"{len(copied)} new image(s):", ", ".join(copied) or "none")

HY21_DIR = os.path.join(WORK, "Hunyuan3D-2.1")
MODEL_ROOT = os.path.join(WORK, "models")
SETTINGS = dict(
    drive_root=DRIVE_ROOT, hf_home=os.path.join(WORK, "hf_cache"), rembg_home=os.path.join(WORK, "rembg_cache"),
    hunyuan_repo=os.path.join(HY21_DIR, "hy3dshape"), model_root=MODEL_ROOT,
    shape_steps=SHAPE_STEPS, shape_guidance=SHAPE_GUIDANCE, shape_seed=SHAPE_SEED,
    deadline=SESSION_START + STOP_AFTER_HOURS * 3600,
)
if QUALITY == "high":
    SETTINGS["presets"] = imageto3d.config.HIGH_QUALITY_PRESETS
probe = imageto3d.config.Config(**SETTINGS)
waiting = [a.key for a in imageto3d.assets.pending(probe) if a.category in CATEGORIES]
skipped = [a.key for a in imageto3d.assets.pending(probe) if a.category not in CATEGORIES]
print(f"{len(waiting)} image(s) for the TPU:", ", ".join(waiting) or "none")
if skipped:
    print(f"Left for the 2x T4 notebook (category not in CATEGORIES): {', '.join(skipped)}")

In [ ]:
# 3. Install Hunyuan3D-2.1 (shape part), download its weights, check imports
if "SETTINGS" not in globals():
    raise RuntimeError("Run blocks 1 and 2 first.")
import os, subprocess, sys

MARK = os.path.join(WORK, ".hy21_shape_tpu_v1")
CONSTRAINTS = os.path.join(WORK, "constraints.txt")
PINNED = [
    "transformers==4.57.3", "diffusers==0.34.0", "huggingface_hub==0.36.0", "accelerate==1.10.1",
    "timm==1.0.29", "omegaconf==2.3.0", "einops==0.8.1", "pymeshlab==2025.7.post1", "trimesh==4.8.3",
    "scikit-image", "opencv-python-headless", "onnxruntime==1.22.1", "rembg==2.0.67", "pygltflib==1.16.5",
    "hf_xet",
]
for d in (SETTINGS["hf_home"], SETTINGS["rembg_home"], MODEL_ROOT):
    os.makedirs(d, exist_ok=True)

if not os.path.exists(MARK):
    with open(CONSTRAINTS, "w") as f:
        f.write("torch==2.8.0\ntorchvision==0.23.0\ntorch_xla==2.8.0\n")
    sh(f"{sys.executable} -m pip install -q -c {CONSTRAINTS} " + " ".join(f"'{p}'" for p in PINNED))
    if not os.path.exists(HY21_DIR):
        # Only the shape code; the repository's demo assets are large.
        sh(f"git init -q {HY21_DIR}")
        sh("git remote add origin https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git", cwd=HY21_DIR)
        sh("git sparse-checkout set hy3dshape", cwd=HY21_DIR)
    sh(f"git fetch -q --depth 1 --filter=blob:none origin {HY21_COMMIT} && git checkout -q FETCH_HEAD", cwd=HY21_DIR)
    open(MARK, "w").close()

env = dict(os.environ, HF_HOME=SETTINGS["hf_home"], U2NET_HOME=SETTINGS["rembg_home"])
# Weights (6.6 GB) and the background-removal model, downloaded once here instead of by every worker.
download = f"""
from huggingface_hub import snapshot_download
snapshot_download("tencent/Hunyuan3D-2.1", allow_patterns=["hunyuan3d-dit-v2-1/*"],
                  local_dir={os.path.join(MODEL_ROOT, "tencent", "Hunyuan3D-2.1")!r})
from rembg import new_session
new_session("isnet-general-use")
print("weights ready")
"""
subprocess.run([sys.executable, "-c", download], check=True, env=env)

# Import everything the workers use, without touching the TPU.
check = ("import sys; sys.path[:0] = [%r, %r]; import torch, torch_xla, hy3dshape, imageto3d.stage_shape_tpu; "
         "from hy3dshape.pipelines import export_to_trimesh; print('torch', torch.__version__, 'imports OK')"
         % (PIPELINE_DIR, SETTINGS["hunyuan_repo"]))
result = subprocess.run([sys.executable, "-c", check], capture_output=True, text=True, env=env)
print(result.stdout, result.stderr[-3000:])
if result.returncode != 0:
    os.remove(MARK)
    raise RuntimeError("Import check failed. Send the last 30 lines above for diagnosis.")

CONFIG_PATH = os.path.join(WORK, "imageto3d_shape_config.json")
imageto3d.config.Config(**SETTINGS).save(CONFIG_PATH)
print("Install OK. Config written to", CONFIG_PATH)

In [ ]:
# 4. Build shapes on the TPU (one worker per chip) and pack the results
if "CONFIG_PATH" not in globals() or not os.path.exists(CONFIG_PATH):
    raise RuntimeError("Run blocks 1, 2 and 3 first.")
import glob, json, os, subprocess, sys, time, zipfile

env = dict(os.environ, PJRT_DEVICE="TPU", PYTHONPATH=os.pathsep.join([PIPELINE_DIR, SETTINGS["hunyuan_repo"]]),
           HF_HOME=SETTINGS["hf_home"], U2NET_HOME=SETTINGS["rembg_home"], TOKENIZERS_PARALLELISM="false")
command = [sys.executable, "-u", "-m", "imageto3d.stage_shape_tpu", "--config", CONFIG_PATH,
           "--categories", *CATEGORIES, "--workers", str(WORKERS),
           "--xla_cache", os.path.join(WORK, "xla_cache"), "--code_commit", HY21_COMMIT]
started = time.time()
process = subprocess.Popen(command, env=env, cwd=PIPELINE_DIR, text=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in process.stdout:
    if "it/s]" not in line and "s/it]" not in line and "B/s]" not in line:
        print(line, end="", flush=True)
process.wait()
print(f"\nShape process exited with code {process.returncode} after {(time.time() - started) / 60:.1f} min.")

work_dir = os.path.join(DRIVE_ROOT, "work")
for leftover in glob.glob(os.path.join(work_dir, "*", "claim.lock")) + glob.glob(os.path.join(work_dir, ".load.lock")):
    os.remove(leftover)
done = sorted(os.path.basename(os.path.dirname(p)) for p in glob.glob(os.path.join(work_dir, "*", "shape_import.json")))
for status_path in glob.glob(os.path.join(work_dir, "*", "status.json")):
    status = json.load(open(status_path))
    if status.get("last_error"):
        print("Problem:", os.path.basename(os.path.dirname(status_path)), status["last_error"][-500:])

archive = f"/kaggle/working/hunyuan3d-shapes-{time.strftime('%Y%m%d-%H%M')}.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as bundle:
    for key in done:
        for name in ("front.png", "shape.glb", "shape_import.json"):
            bundle.write(os.path.join(work_dir, key, name), f"work/{key}/{name}")
    for name in sorted(os.listdir(os.path.join(DRIVE_ROOT, "in"))):
        bundle.write(os.path.join(DRIVE_ROOT, "in", name), f"in/{name}")
print(f"Done. {len(done)} shape(s): {', '.join(done) or 'none'}")
print(f"Packed into {archive}.")
print("Next: in hunyuan3d2_kaggle_2xt4.ipynb, Add Input > this notebook's output, then run it to paint and export.")

In [ ]:
# 5. Review the shapes (optional): front image and a shaded front/side view of each mesh
import glob, os
import numpy as np, trimesh
from PIL import Image, ImageDraw
from IPython.display import display

def shaded(mesh, axis, size=320):
    """Flat-shaded orthographic view along -Z (axis="front") or -X (axis="side"), painter's algorithm."""
    v = mesh.vertices.copy()
    if axis == "side":
        v = v[:, [2, 1, 0]] * np.array([-1, 1, 1])
    tris = v[mesh.faces]
    normals = np.cross(tris[:, 1] - tris[:, 0], tris[:, 2] - tris[:, 0])
    normals /= np.linalg.norm(normals, axis=1, keepdims=True) + 1e-12
    light = np.clip(normals @ np.array([0.3, 0.4, 0.87]), 0, 1)
    lo, hi = v[:, :2].min(0), v[:, :2].max(0)
    scale = 0.9 * size / (hi - lo).max()
    xy = (tris[:, :, :2] - (lo + hi) / 2) * scale + size / 2
    xy[:, :, 1] = size - xy[:, :, 1]
    image = Image.new("RGB", (size, size), (40, 40, 40))
    draw = ImageDraw.Draw(image)
    for i in np.argsort(tris[:, :, 2].mean(1)):
        shade = int(60 + 190 * light[i])
        draw.polygon([tuple(p) for p in xy[i]], fill=(shade, shade, shade))
    return image

for work in sorted(glob.glob(os.path.join(DRIVE_ROOT, "work", "*", "shape.glb"))):
    key = os.path.basename(os.path.dirname(work))
    mesh = trimesh.load(work, force="mesh")
    front = Image.open(os.path.join(os.path.dirname(work), "front.png")).convert("RGBA")
    front.thumbnail((320, 320))
    tile = Image.new("RGB", (960, 320), (40, 40, 40))
    tile.paste(front, ((320 - front.width) // 2, (320 - front.height) // 2), front)
    tile.paste(shaded(mesh, "front"), (320, 0))
    tile.paste(shaded(mesh, "side"), (640, 0))
    print(f"{key}: {len(mesh.faces):,} faces")
    display(tile)